In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/jilmedavidesco@gmail.com/pipeline_project/1_code/1_setup/2_utilities

In [0]:
dbutils.widgets.text("catalog", "atlone_enterprise", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
# Leer tabla oro de la compañia padre
df_parent = spark.table(f"{catalog}.{gold_schema}.parent_dim_{data_source}")

# Crear tabla estadarizada "dim_customers"
df_parent.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.1_dim_{data_source}")

In [0]:
# Merge child

# Leer tabla oro de la compañia hija
df_child = spark.sql(f"SELECT customer_id AS customer_code, customer, market, platform, channel FROM {catalog}.{gold_schema}.child_dim_{data_source};")

delta_table = DeltaTable.forName(spark, "atlone_enterprise.gold.1_dim_customers")

# Merge con tabla estadarizada "dim_customers"
delta_table.alias("target").merge(
    source=df_child.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()